# ⚙️ Der Optimizer ist dein Compiler

Traditionell: **Quellcode → Compiler → Binary**
Heute: **Signature + Metrik + Daten → Der Optimizer → Optimierter Prompt**

Beides nimmt menschenlesbare Spezifikationen und produziert maschinenausführbare Artefakte.

Im letzten Notebook hast du gesehen: selbst wenn du dem Modell die gültigen Kategorien, Prioritäten und Gruppennamen gibst, bleibt der Score bei ~30%. Kann ein **automatischer Optimizer** das besser?

## Was passiert hier?

Im letzten Notebook hast du Prompts **manuell** verbessert. Jetzt lässt du den **Computer** das machen.

Das Prinzip ist einfach:
1. Du sagst, **was** du willst (Signature)
2. Du sagst, **was gut heisst** (Metrik)
3. Du gibst **Beispiele** (Daten)
4. Der Optimizer findet automatisch den besten Prompt

Das ist wie ein Compiler: Du schreibst Quellcode, der Compiler macht eine optimierte Binary. Hier schreibst du eine Spezifikation, der Optimizer macht einen optimierten Prompt.


In [1]:
import sys
sys.path.insert(0, ".")
from dspy_tasks.config import get_available_models, configure_dspy

# Verfügbare Modelle
print("Verfügbare Modelle:")
for m in get_available_models()[:10]:
    print(f"  • {m}")

# Modell wählen (ändere den String um ein anderes zu nutzen)
MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)
print(f"\n✅ Konfiguriert: {MODEL}")

Verfügbare Modelle:
  • github_copilot/gpt-4o
  • github_copilot/claude-sonnet-4
  • github_copilot/gpt-4o-mini
  • github_copilot/gpt-41-copilot
  • github_copilot/gpt-5.2
  • github_copilot/claude-haiku-4.5
  • github_copilot/text-embedding-3-small-inference
  • github_copilot/claude-opus-4.5
  • github_copilot/gpt-3.5-turbo-0613
  • github_copilot/gpt-5

✅ Konfiguriert: github_copilot/gpt-5.1


### 🔄 Der Optimizer-Workflow als Diagramm

So sieht automatische Optimierung aus — in vier Schritten. Du lieferst die **Zutaten** (Signature, Metrik, Daten), der Optimizer **kocht** daraus den besten Prompt.

Das Schöne: Wenn sich deine Daten ändern, optimierst du einfach nochmal. Der Prozess ist reproduzierbar.


In [4]:
from dspy_tasks.visualize import diagram

diagram([
    {"label": "Signature", "detail": "Was du willst", "icon": "📝", "color": "#0078d4"},
    {"label": "Metrik", "detail": "Was 'gut' heisst", "icon": "📐", "color": "#0078d4"},
    {"label": "Trainingsdaten", "detail": "Beispiele", "icon": "📊", "color": "#0078d4"},
    {"label": "Optimizer", "detail": "probiert Varianten", "icon": "⚙️", "color": "#ca5010"},
    {"label": "Optimierter Prompt", "detail": "+ Few-Shot Demos", "icon": "🎯", "color": "#107c10"},
], title="Die Optimierungs-Pipeline")

## ✏️ Erst du, dann die Maschine

Bevor wir den automatischen Optimizer loslassen, versuch es nochmal selbst! Das ist dieselbe Ticket-Routing-Aufgabe aus Notebook 01. 

Editiere den Prompt unten und schau, welchen Score du erreichst. Merk dir deinen besten Score — danach vergleichen wir mit dem automatisch optimierten Ergebnis.

### 🤔 Warum erst manuell?

Gute Frage! Wir wollen, dass du ein **Gefühl** für Prompt-Engineering bekommst. Wenn du selbst versucht hast, einen Prompt zu verbessern, verstehst du viel besser, was der Optimizer tut — und warum er es besser kann als wir.

Merk dir deinen Score — gleich vergleichen wir ihn mit dem des Optimizers.


In [2]:
from dspy_tasks.actions import run_with_prompt
from dspy_tasks.visualize import display_score, display_results_table

# Dein manueller Versuch — ändere den Prompt und führe die Zelle erneut aus!
MEIN_PROMPT = """Classify this IT support ticket.
Category MUST be one of: Event NO Customer Impact, Failure, Service Request
Priority MUST be one of: High, Medium, Standard
Assigned group MUST be one of: SDE - Service Desk, OFC - Office & Collaboration, PRM - Premium Support, CDC - Client Design & Standard SW Integration, OUM - Incident, Helpdesk, Service-Center IKT, Service-Center IKT Bestellungen, Smartcard Office, CBCD - Container Basierte Cloud Dienste, OPC - Applikationen, DevOps - Rein, BVX - ePortal Service Line, IOM - Input / Output Mgmt., OPM - DLC Dispatching, O-SDK, ESTV-RSS-Stammdaten, FIB - BIT Store Bollwerk, Immobilien BAZG, Bedarfsmanagement, Güter und Ausrüstung"""

result = run_with_prompt("ticket_routing", MEIN_PROMPT, max_eval=3)
display_score("Dein manueller Prompt", result.score)
display_results_table(result.individual_scores)

  [1/3] ✓
  [2/3] ✓
  [3/3] ✓


## ⚙️ Und jetzt die Maschine...

Du hast deinen besten manuellen Score gesehen. Selbst MIT allen gültigen Werten im Prompt liegt er vermutlich bei 30-50%. Das Modell kennt die Werte, aber es weiss nicht, **wann welcher Wert passt**.

Jetzt drück unten auf "Optimieren" und schau, was passiert. Der Optimizer lernt aus den Trainingsdaten die **Zuordnungsregeln** und findet automatisch den besten Prompt.

**Die Frage ist:** Kann der Computer einen besseren Prompt finden als du?

## BootstrapFewShot: Der schnelle Compiler

**BootstrapFewShot** ist wie `-O1` Optimierung — schnell und effektiv. Er sucht die besten Few-Shot-Beispiele aus deinen Trainingsdaten und fügt sie in den Prompt ein.

Beim Ticket-Routing heisst das: Der Optimizer wählt automatisch die informativsten Beispiel-Tickets aus, damit das Modell lernt: *"Account gesperrt" → SDE - Service Desk, "CPAM Ausfall" → CDC*.

Dauer: ~10 Sekunden. Verbesserung: oft 10-30%.

### 📋 Was macht BootstrapFewShot genau?

Stell dir vor, du hast 35 Trainings-Tickets. BootstrapFewShot probiert verschiedene Kombinationen durch und findet heraus: *Welche 3-5 Beispiel-Tickets im Prompt liefern die besten Ergebnisse?*

Es ist wie ein Koch, der verschiedene Gewürzkombinationen testet — systematisch statt nach Bauchgefühl.

In [3]:
from dspy_tasks.actions import run_optimization, run_with_prompt
from dspy_tasks.tasks import get_task
from dspy_tasks.visualize import display_improvement, display_insight, display_prompt_diff, display_score, display_results_table

task = get_task("ticket_routing")
print(f"⏳ Optimiere {task.name} mit BootstrapFewShot...")
print(f"   Das kann 10-60 Sekunden dauern...\n")

result = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8)

# Zeige Baseline (zero-shot, OHNE Prompt) vs. Optimiert
print("━" * 60)
print("📊 VORHER: Zero-Shot Baseline (kein Prompt, kein Beispiel)")
display_score("Zero-Shot Baseline", result.baseline_score)
if result.baseline_individual_scores:
    display_results_table(result.baseline_individual_scores)

print("\n" + "━" * 60)
print("📊 NACHHER: Optimierter Prompt (mit Few-Shot Beispielen)")
display_score("Nach BootstrapFewShot", result.optimized_score)
if result.optimized_individual_scores:
    display_results_table(result.optimized_individual_scores)

display_improvement(result.baseline_score, result.optimized_score)
print(f"⏱️  Optimierung dauerte {result.elapsed_seconds}s | {result.llm_calls} LLM-Aufrufe")

display_prompt_diff(result.prompt_before, result.prompt_after)

display_insight("Was gerade passiert ist",
    f"Zero-Shot (ohne Prompt): {result.baseline_score:.0%}. "
    f"Nach Optimierung: {result.optimized_score:.0%}. "
    "Der Optimizer hat aus den Trainingsdaten die besten Beispiel-Tickets ausgewählt und in den Prompt eingefügt. "
    "Vergleich das mit deinem manuellen Prompt oben — ist der Optimizer besser?")

⏳ Optimiere Ticket Classification & Routing mit BootstrapFewShot...
   Das kann 10-60 Sekunden dauern...

  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


 14%|█▍        | 5/35 [00:00<00:00, 46.71it/s]

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] 

✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 VORHER: Zero-Shot Baseline (kein Prompt, kein Beispiel)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 NACHHER: Optimierter Prompt (mit Few-Shot Beispielen)


⏱️  Optimierung dauerte 0.12s | 16 LLM-Aufrufe


## MIPROv2: Der Heavy-Duty Compiler

**MIPROv2** ist wie `-O3` — er optimiert gleichzeitig die **Instruktionen UND die Beispiele** mit Bayesian Search. Während BootstrapFewShot nur Beispiele auswählt, schreibt MIPROv2 den Prompt-Text selbst um.

Dauer: ~30-60 Sekunden. Verbesserung: oft nochmal besser als BootstrapFewShot.

### 🔀 BootstrapFewShot vs. MIPROv2 — der direkte Vergleich

Zwei verschiedene Strategien treten gegeneinander an:

| Optimizer | Was er optimiert | Stärke |
|---|---|---|
| **BootstrapFewShot** | Welche Beispiele im Prompt stehen | Schnell, zuverlässig |
| **MIPROv2** | Beispiele UND Anweisungen | Gründlicher, aber langsamer |

Welcher gewinnt? Klick den Button und finde es heraus. Spoiler: Es hängt vom Task ab!


In [ ]:
from dspy_tasks.visualize import bar_comparison

task = get_task("ticket_routing")

# --- BootstrapFewShot ---
print(f"⏳ BootstrapFewShot auf {task.name}...\n")
r_bs = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8)
print(f"\n✅ BootstrapFewShot: {r_bs.baseline_score:.0%} → {r_bs.optimized_score:.0%}")
display_improvement(r_bs.baseline_score, r_bs.optimized_score)

print("\n📝 BootstrapFewShot — Prompt VORHER:")
print(r_bs.prompt_before)
print("\n📝 BootstrapFewShot — Prompt NACHHER:")
print(r_bs.prompt_after[:2000])
if len(r_bs.prompt_after) > 2000:
    print(f"... ({len(r_bs.prompt_after)} Zeichen)")

print("\n📊 BootstrapFewShot — Ergebnisse pro Ticket:")
if r_bs.optimized_individual_scores:
    display_results_table(r_bs.optimized_individual_scores)

# --- MIPROv2 ---
print(f"\n{'━'*60}")
print(f"⏳ MIPROv2 auf {task.name}...\n")
r_mipro = run_optimization("ticket_routing", "MIPROv2", max_eval=8)
print(f"\n✅ MIPROv2: {r_mipro.baseline_score:.0%} → {r_mipro.optimized_score:.0%}")
display_improvement(r_mipro.baseline_score, r_mipro.optimized_score)

print("\n📝 MIPROv2 — Prompt VORHER:")
print(r_mipro.prompt_before)
print("\n📝 MIPROv2 — Prompt NACHHER:")
print(r_mipro.prompt_after[:2000])
if len(r_mipro.prompt_after) > 2000:
    print(f"... ({len(r_mipro.prompt_after)} Zeichen)")

print("\n📊 MIPROv2 — Ergebnisse pro Ticket:")
if r_mipro.optimized_individual_scores:
    display_results_table(r_mipro.optimized_individual_scores)

# --- Vergleich ---
print(f"\n{'━'*60}")
print("📊 Zusammenfassung:")
print(f"   Dein manueller Prompt: scroll hoch und vergleich!")
print(f"   Zero-Shot Baseline:     {r_bs.baseline_score:.0%}")
print(f"   BootstrapFewShot:       {r_bs.optimized_score:.0%}")
print(f"   MIPROv2:                {r_mipro.optimized_score:.0%}")

scores = {
    "BootstrapFewShot": {"baseline": r_bs.baseline_score, "optimized": r_bs.optimized_score},
    "MIPROv2": {"baseline": r_mipro.baseline_score, "optimized": r_mipro.optimized_score},
}
fig = bar_comparison("Ticket Routing: Optimizer-Vergleich", scores)
fig.show()

⏳ BootstrapFewShot auf Ticket Classification & Routing...

  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


 14%|█▍        | 5/35 [00:00<00:00, 63.53it/s]

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓


  [7/8] ✓
  [8/8] ✓

✅ BootstrapFewShot: 5% → 54%



📝 BootstrapFewShot — Prompt VORHER:
(zero-shot)

📝 BootstrapFewShot — Prompt NACHHER:
{'predict': {'traces': [], 'train': [], 'demos': [{'augmented': True, 'summary': 'Account gesperrt', 'reasoning': '"Account gesperrt" indicates a locked user account, similar to the previous example "Der Benutzeraccount ist gesperrt." That type of issue is treated as a failure and handled by the central service center.', 'category': 'Failure', 'priority': 'Standard', 'assigned_group': 'Service-Center IKT'}, {'augmented': True, 'summary': 'Smartcard Force Logon aufheben/Smartcard vergessen', 'reasoning': 'Issue with smartcard logon configuration / access (force logon removal or forgotten smartcard) is an access/authentication problem typically handled as a standard service request by the service desk.', 'category': 'Service Request', 'priority': 'Standard', 'assigned_group': 'SDE - Service Desk'}, {'augmented': True, 'summary': 'CPAM Ausfall', 'reasoning': '"CPAM Ausfall" (CPAM outage/failure) indicat

...

2026/03/24 17:27:33 INFO dspy.evaluate.evaluate: Average Metric: 3.4 / 28 (12.1%)


2026/03/24 17:27:33 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 12.14 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5'].
2026/03/24 17:27:33 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [1.43, 10.71, 7.14, 10.71, 10.71, 12.14, 7.14, 11.96, 9.29, 11.96, 12.14]
2026/03/24 17:27:33 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 12.14
2026/03/24 17:27:33 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/24 17:27:33 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 12.14!


  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓

✅ MIPROv2: 5% → 11%



📝 MIPROv2 — Prompt VORHER:
(zero-shot)

📝 MIPROv2 — Prompt NACHHER:
{'predict': {'traces': [], 'train': [], 'demos': [{'augmented': True, 'summary': 'Microsoft Windows - Mail als PDF speichern übernimmt Bilder nicht', 'reasoning': 'User reports an issue with Microsoft Windows where saving an email as PDF does not include embedded images. This is an application/software malfunction, not a hardware or access issue. It impacts functionality but is not business‐critical, so priority is Medium. First-level application support / service desk is appropriate.', 'category': 'Failure', 'priority': 'Medium', 'assigned_group': 'SDE - Service Desk'}, {'augmented': True, 'summary': 'CPAM Ausfall', 'reasoning': '"CPAM Ausfall" indicates an outage/failure of a specific application or system (CPAM). An outage is an incident (failure) rather than a normal service request and likely affects multiple users, so it should be treated with elevated urgency and handled by the general service desk or incident-


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


### 🤔 Food for Thought: Warum bleibt der Score unter 100%?

Selbst nach Optimierung erreichen wir selten 100%. Warum?

**Das Kernproblem:** Das Modell muss aus einem kurzen Ticket-Text auf 3 Felder schliessen — und die Zuordnungsregeln sind **nirgends explizit**.

- Warum geht "VPN funktioniert nicht" an **CDC** und nicht an **OFC**? Das steht in keinem Handbuch — es ist **organisatorisches Wissen**.
- Warum ist "Drucker geht nicht" **Standard** und "SAP-Ausfall" **High**? Das sind **implizite Regeln** aus jahrelanger Praxis.

**Wie könnte man das verbessern?**

1. **Mehr Trainingsdaten** — 35 Beispiele sind wenig. Mit 200+ Tickets lernt der Optimizer die Zuordnungsregeln viel besser.
2. **Bessere Beschreibung** — Die Ticket-Texte sind oft kurz und mehrdeutig. Mehr Kontext (z.B. Abteilung, System) würde helfen.
3. **Domain-spezifische Regeln im Prompt** — z.B. "Alle VPN-Probleme gehen an CDC", "SAP-Tickets sind immer High Priority".
4. **Fine-Tuning** — Statt nur den Prompt zu optimieren, das Modell selbst auf die Daten trainieren.

> 💡 **Die Erkenntnis:** Der Optimizer kann nur so gut sein wie die Daten. Er findet die besten *vorhandenen* Muster — aber er kann kein Wissen erfinden, das nicht in den Beispielen steckt. **Deine Daten sind dein Burggraben** — genau das Thema des nächsten Notebooks.

## 🎯 Beliebige Aufgabe optimieren

Wähl eine Aufgabe aus der Liste unten und sieh dir an, was der Optimizer daraus macht. Jede Aufgabe hat eine andere Schwierigkeit — und der Optimizer zeigt dir genau, was er am Prompt ändert.

**So geht's:** Ändere den `TASK`-String und führe die Zelle aus. Der Prompt-Diff zeigt dir den Unterschied.

In [4]:
# Verfügbare Aufgaben zum Optimieren:
available = ["ticket_routing", "multihop_qa", "report_generation"]
print("Verfügbare Aufgaben:")
for tid in available:
    t = get_task(tid)
    examples = t.load_examples()[:2]
    print(f"\n  📋 {tid}: {t.name}")
    print(f"     {t.description}")
    if examples:
        sample = examples[0]
        input_preview = " | ".join(f"{k}={str(v)[:60]}" for k, v in sample.inputs().items())
        print(f"     Beispiel-Input: {input_preview[:120]}...")

# ═══ Wähle eine Aufgabe (ändere den String): ═══
TASK = "ticket_routing"

print(f"\n{'━'*60}")
print(f"⏳ Optimiere {TASK}...")
result = run_optimization(TASK, "BootstrapFewShot", max_eval=8)
display_improvement(result.baseline_score, result.optimized_score)
display_prompt_diff(result.prompt_before, result.prompt_after)

# Zeige Ergebnisse pro Beispiel
print("\n📊 Ergebnisse nach Optimierung:")
if result.optimized_individual_scores:
    display_results_table(result.optimized_individual_scores)

Verfügbare Aufgaben:

  📋 ticket_routing: Ticket Classification & Routing
     Classify IT support tickets by category, priority, and routing team.
     Beispiel-Input: summary=Account gesperrt...

  📋 multihop_qa: Multi-Hop QA
     Answer questions requiring 2-3 reasoning hops across a passage.
     Beispiel-Input: question=What is the capital of the country where the Danube River en | context=The Danube River flows through ten count...

  📋 report_generation: Structured Report Generation
     Generate structured reports from data points.
     Beispiel-Input: data_points=Team: SDE - Service Desk, Total tickets: 601, Priority distr | report_type=team_summary...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⏳ Optimiere ticket_routing...
  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


 14%|█▍        | 5/35 [00:00<00:00, 47.64it/s]

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] 

✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓



📊 Ergebnisse nach Optimierung:


## 🏆 Vergleich: Du vs. Maschine

Scroll zurück und vergleich die Ergebnistabellen:
- **Dein manueller Prompt** (oben): Schau dir an, welche Tickets du richtig hattest
- **BootstrapFewShot**: Hat er die gleichen Fehler gemacht? Oder andere?
- **MIPROv2**: Wo hat der umgeschriebene Prompt geholfen?

Der Unterschied zeigt sich in den **Details**:
- Der Optimizer wählt Beispiel-Tickets, die dem Modell die **Zuordnungsregeln** beibringen
- MIPROv2 schreibt zusätzlich die Anweisungen um — oft in einer Art, die du selbst nicht probiert hättest
- Aber beide Optimizer können nur Muster finden, die **in den Trainingsdaten vorhanden sind**

> **Das ist der Punkt:** Manuelles Prompt-Tuning funktioniert, ist aber langsam und fragil. Automatische Optimierung findet bessere Prompts, schneller, reproduzierbar. **Deine Metrik + Daten = dein Programm. Der Optimizer ist der Compiler.**

## ⏭️ Weiter geht's!

Der Optimizer hat den Prompt verbessert — automatisch, messbar, reproduzierbar. Aber was passiert, wenn du **deine eigenen, echten Daten** benutzt?

Im nächsten Notebook nehmen wir die echten Ticket-Daten aus dem Projekt und zeigen: **Deine Daten sind dein Burggraben.** Ein generisches Modell + deine Domain-Daten + Tuning = etwas, das kein Konkurrent kopieren kann.

👉 **[Weiter zu Notebook: Domain-Tuning →](03_domain_tuning.ipynb)**
